In [ ]:
%pip install -q deep-translator bert-score sentence-transformers nltk spacy

In [ ]:
import time
import importlib
from deep_translator import MyMemoryTranslator
import nltk
import spacy
from nltk.corpus import wordnet as wn

nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("omw-2.0")

In [ ]:


def traduzir_frase(frase_original):
    frase = MyMemoryTranslator(source='portuguese', target='japanese').translate(frase_original)
    time.sleep(1)
    frase_volta_portugues = MyMemoryTranslator(source='japanese', target='portuguese').translate(frase)
    time.sleep(1)
    return frase_volta_portugues

def read_frases_from_file(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]



In [11]:

nlp = spacy.load("pt_core_news_sm")


def contradizer(frase):
    doc = nlp(frase)
   
    if doc and doc[0].pos_ == "PROPN":
        doc = nlp(frase[:1].lower() + frase[1:])

    raiz = None
    for palavra in doc:
        if palavra.dep_ == "ROOT":
            raiz = palavra

    if raiz is None:
        return None

    for palavra in raiz.children:
        if palavra.lower_ == "não":
            nova = ""
            for p in doc:
                if p != palavra:
                    nova += p.text_with_ws
            return nova[0].upper() + nova[1:]


    alvo = None
    for palavra in raiz.children:
        if palavra.dep_ in ["aux", "aux:pass", "cop"] and palavra.i < raiz.i:
            alvo = palavra
            break

    if alvo is None:
        if raiz.pos_ == "VERB" or raiz.pos_ == "AUX":
            alvo = raiz
        else:
            return None

    if alvo.i == 0:
        return "Não " + frase[0].lower() + frase[1:]
    return frase[:alvo.idx] + "não " + frase[alvo.idx:]


In [12]:
def main():
    frases = read_frases_from_file('frases.txt')
    frases_parafraseada = []
    frases_contradicadas = []

    for frase in frases:
        frases_contradicadas.append(contradizer(frase))
        frases_parafraseada.append(traduzir_frase(frase))
        
    print(frases_parafraseada)
    print(frases_contradicadas)


    with open('frases_parafraseadas.txt', 'w', encoding='utf-8') as f:
        for frase in frases_parafraseada:
            f.write(frase + '\n')

    # contradizer retorna None quando não acha verbo; linha vazia mantém o alinhamento com frases.txt
    with open('frases_contradicadas.txt', 'w', encoding='utf-8') as f:
        for frase in frases_contradicadas:
            f.write((frase or '') + '\n')


main()

['O sol nasceu cedo e iluminou toda a cidade com uma luz dourada.', 'Gosta de tomar café forte pela manhã, sempre sem açúcar e muito quente.', 'O cão tem dormido tranquilamente no sofá da sala de estar desde o início da tarde.', 'Vamos ver a estreia do novo filme amanhã à noite.', 'Comprou um carro novo e já está a planear uma viagem de fim de semana.', 'Fortes chuvas caíram no meio da noite e as ruas ficaram completamente inundadas.', 'As crianças se divertem no parquinho enquanto os pais conversam.', 'Estamos aprendendo inteligência artificial com exemplos práticos e divertidos.', 'O entregador se perdeu e a pizza chegou gelada.', 'O ônibus estava cheio e levava mais de uma hora para chegar ao centro.', 'Ela lia três livros por mês e anotava cada um.', 'Durante a tempestade, o vento derrubou as árvores em frente à escola.', 'A aula atrasou porque o professor ficou preso num engarrafamento.', 'A gata subiu no telhado e não consegue mais sair sozinha.', 'Ele deixou a carteira em casa e